# Notebook 07 — QuantFormer Fusion Model

## Objective

In this notebook we build the **QuantFormer Fusion Model**, which combines:

- **Market Representation** extracted from the pretrained Temporal Fusion Transformer (TFT)
- **Financial News Representation** extracted from the pretrained FinBERT model

The objective is to improve stock price movement prediction by leveraging both numerical market data and textual financial news.

---

## Pipeline

FI-2010 Dataset
↓
Pretrained TFT
↓
128-D Market Features

Financial News
↓
Pretrained FinBERT
↓
768-D News Embeddings

Market Features + News Embeddings
↓
QuantFormer Fusion Network
↓
3-Class Prediction

- Down
- Stable
- Up

---

## Models Used

- Temporal Fusion Transformer (Frozen)
- FinBERT (Frozen)
- QuantFormer Fusion Network (Trainable)

Only the Fusion Network is trained in this notebook.

In [51]:
import os
import sys
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import src.project_config as config
from tqdm import tqdm

warnings.filterwarnings("ignore")

In [52]:
# ============================================================
# Project Root
# ============================================================

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("=" * 60)
print("Project Root")
print("=" * 60)
print(PROJECT_ROOT)

Project Root
d:\Coding\Quant Former


In [53]:
print("Project Configuration Imported")

Project Configuration Imported


In [54]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print("Device")
print("=" * 60)
print(DEVICE)

Device
cpu


In [55]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 60)
print("Random Seed Fixed")
print("=" * 60)
print(SEED)

Random Seed Fixed
42


In [56]:
os.makedirs("../checkpoints", exist_ok=True)
os.makedirs("../saved_models", exist_ok=True)
os.makedirs("../artifacts", exist_ok=True)

print("=" * 60)
print("Directories Ready")
print("=" * 60)

Directories Ready


In [57]:
TFT_CHECKPOINT = "../checkpoints/best_tft_model.pth"

FINBERT_MODEL = "../saved_models/best_finbert_model.pth"

EMBEDDING_CACHE = "../artifacts/finbert_embeddings.npy"

FUSION_CHECKPOINT = "../checkpoints/best_fusion_model.pth"

print("=" * 60)
print("Paths")
print("=" * 60)

print("TFT :", TFT_CHECKPOINT)
print("FinBERT :", FINBERT_MODEL)
print("Embeddings :", EMBEDDING_CACHE)
print("Fusion :", FUSION_CHECKPOINT)

Paths
TFT : ../checkpoints/best_tft_model.pth
FinBERT : ../saved_models/best_finbert_model.pth
Embeddings : ../artifacts/finbert_embeddings.npy
Fusion : ../checkpoints/best_fusion_model.pth


In [58]:
files = [
    TFT_CHECKPOINT,
    FINBERT_MODEL
]

print("=" * 60)
print("File Verification")
print("=" * 60)

for file in files:

    if os.path.exists(file):
        print(f"✓ {file}")

    else:
        print(f"✗ {file}")

File Verification
✓ ../checkpoints/best_tft_model.pth
✓ ../saved_models/best_finbert_model.pth


# Section 2 — Load Pretrained Temporal Fusion Transformer (TFT)

## Objective

In this section we load the pretrained **Temporal Fusion Transformer (TFT)** model that was trained in the previous notebook.

The TFT model is **not trained again** in this notebook.

Instead, it is:

- Loaded from the checkpoint
- Switched to evaluation mode
- Frozen (all gradients disabled)

During fusion training, TFT acts only as a **market feature extractor**.

### Input

Market data

```
(Batch Size, 100, 143)
```

### Output

```
Pooled Market Features (128-D)
```

These features will later be combined with FinBERT news embeddings.

In [59]:
# ============================================================
# Import TFT Model
# ============================================================

from src.models.tft_model import TemporalFusionTransformer

print("=" * 60)
print("Temporal Fusion Transformer Imported")
print("=" * 60)

Temporal Fusion Transformer Imported


In [60]:
# ============================================================
# Initialize TFT
# ============================================================

tft_model = TemporalFusionTransformer()

print("=" * 60)
print("TFT Model Created")
print("=" * 60)

print(tft_model.__class__.__name__)

TFT Model Created
TemporalFusionTransformer


In [61]:
# ============================================================
# Initialize TFT
# ============================================================

tft_model = TemporalFusionTransformer()

print("=" * 60)
print("TFT Model Created")
print("=" * 60)

print(tft_model.__class__.__name__)

TFT Model Created
TemporalFusionTransformer


In [62]:
# ============================================================
# Freeze TFT Parameters
# ============================================================

for param in tft_model.parameters():
    param.requires_grad = False

trainable = sum(p.requires_grad for p in tft_model.parameters())
total = sum(1 for _ in tft_model.parameters())

print("=" * 60)
print("Freeze Verification")
print("=" * 60)

print(f"Trainable Parameters : {trainable}")
print(f"Total Parameters     : {total}")

Freeze Verification
Trainable Parameters : 0
Total Parameters     : 52


In [63]:
dummy_market = torch.randn(
    2,
    SEQUENCE_LENGTH,
    NUM_FEATURES
).to(DEVICE)

with torch.no_grad():

    outputs = tft_model(dummy_market)

print("=" * 60)
print("Forward Pass Verification")
print("=" * 60)

if isinstance(outputs, tuple):

    print("Number of Outputs :", len(outputs))

    for i, out in enumerate(outputs):

        if torch.is_tensor(out):
            print(f"Output {i+1} :", out.shape)

else:

    print(outputs.shape)

Forward Pass Verification
Number of Outputs : 3
Output 1 : torch.Size([2, 3])
Output 2 : torch.Size([2, 128])
Output 3 : torch.Size([2, 4, 100, 100])


In [64]:
with torch.no_grad():

    logits, pooled_features, attention = tft_model(dummy_market)

print("=" * 60)
print("Market Feature Verification")
print("=" * 60)

print("Logits          :", logits.shape)
print("Market Features :", pooled_features.shape)
print("Attention       :", attention.shape)


Market Feature Verification
Logits          : torch.Size([2, 3])
Market Features : torch.Size([2, 128])
Attention       : torch.Size([2, 4, 100, 100])


# Section 3 — Load Pretrained FinBERT

## Objective

In this section we load the pretrained **FinBERT** model, which extracts semantic representations from financial news.

The model was fine-tuned previously and will be used only as a **feature extractor**.

During fusion training:

- FinBERT remains frozen.
- Only the Fusion Network is trained.
- Each news headline is converted into a **768-dimensional embedding**.

These embeddings will later be fused with the TFT market representation.

In [65]:
from src.models.finbert_model import FinBERTExtractor

print("=" * 60)
print("FinBERT Imported")
print("=" * 60)

FinBERT Imported


In [66]:
finbert = FinBERTExtractor(device=config.DEVICE)

print("=" * 60)
print("FinBERT Loaded Successfully")
print("=" * 60)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 28419.98it/s]
[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT Loaded Successfully


In [67]:
for param in finbert.model.parameters():
    param.requires_grad = False

finbert.model.eval()

trainable = sum(p.requires_grad for p in finbert.model.parameters())
total = sum(1 for _ in finbert.model.parameters())

print("=" * 60)
print("Freeze Verification")
print("=" * 60)

print(f"Trainable Parameters : {trainable}")
print(f"Total Parameters     : {total}")

Freeze Verification
Trainable Parameters : 0
Total Parameters     : 199


In [68]:
sample_news = (
    "Federal Reserve keeps interest rates unchanged while "
    "markets react positively."
)

embedding = finbert.extract_embedding(sample_news)

print("=" * 60)
print("Embedding Verification")
print("=" * 60)

print(type(embedding))
print(embedding.shape)

Embedding Verification
<class 'torch.Tensor'>
torch.Size([768])


# Section 4 — Generate / Load Cached FinBERT Embeddings

## Objective

Generating FinBERT embeddings for every training epoch is computationally expensive.

To improve training efficiency, we precompute the embeddings once and store them on disk.

During future training runs:

- If cached embeddings already exist, they are loaded directly.
- Otherwise, embeddings are generated using the pretrained FinBERT model and saved for future use.

This significantly reduces the Fusion model training time.

In [69]:
import numpy as np
import pandas as pd
from tqdm import tqdm

In [70]:
import os

print("PhraseBank Directory:")
print(config.PHRASEBANK_DIR)

print("\nFiles:")

if os.path.exists(config.PHRASEBANK_DIR):
    print(os.listdir(config.PHRASEBANK_DIR))
else:
    print("Directory not found.")

PhraseBank Directory:
d:\Coding\Quant Former\datasets\processed\PhraseBank

Files:
['train.pt', 'val.pt']


# Section 4 — Load Cached FinBERT Embeddings

## Objective

FinBERT embeddings were generated during the previous stage of the pipeline and stored inside the artifacts directory.

Instead of generating embeddings again, we simply load the cached embeddings for Fusion model training.

This greatly reduces training time because FinBERT inference is performed only once.

In [73]:
import importlib
import src.project_config as config

importlib.reload(config)

<module 'src.project_config' from 'd:\\Coding\\Quant Former\\src\\project_config.py'>

In [74]:
print(config.TRAIN_FINBERT_EMBEDDINGS)
print(config.VAL_FINBERT_EMBEDDINGS)

d:\Coding\Quant Former\artifacts\train_finbert_embeddings.npy
d:\Coding\Quant Former\artifacts\val_finbert_embeddings.npy


In [75]:
# ============================================================
# Section 4 : Load Cached FinBERT Embeddings
# ============================================================

import numpy as np
import torch

train_embedding_path = config.TRAIN_FINBERT_EMBEDDINGS
val_embedding_path = config.VAL_FINBERT_EMBEDDINGS

train_embeddings = np.load(train_embedding_path)
val_embeddings = np.load(val_embedding_path)

train_finbert_embeddings = torch.tensor(
    train_embeddings,
    dtype=torch.float32
).to(config.DEVICE)

val_finbert_embeddings = torch.tensor(
    val_embeddings,
    dtype=torch.float32
).to(config.DEVICE)

print("=" * 60)
print("Train Embeddings :", train_finbert_embeddings.shape)
print("Validation Embeddings :", val_finbert_embeddings.shape)
print("Device :", config.DEVICE)

Train Embeddings : torch.Size([1807, 768])
Validation Embeddings : torch.Size([452, 768])
Device : cpu


# Section 5 — Build QuantFormer Fusion Model

## Objective

In this section, we initialize the complete QuantFormer Fusion architecture.

The Fusion model combines:

- Market representations extracted by the frozen TFT encoder.
- Financial news embeddings extracted by FinBERT.

Both modalities are projected into a common latent space and fused through the QuantFormer Fusion Network.

The final classifier predicts the next market movement into one of three classes:

- DOWN
- STABLE
- UP

During Fusion training, the TFT encoder remains frozen while only the fusion layers and classifier are optimized.

In [76]:
from src.fusion.fusion_model import QuantFormerFusion

print("=" * 60)
print("Fusion Model Imported Successfully")
print("=" * 60)

Fusion Model Imported Successfully


In [77]:
fusion_model = QuantFormerFusion(
    tft_model=tft_model
).to(config.DEVICE)

print("=" * 60)
print("QuantFormer Fusion Model Initialized")
print("=" * 60)

QuantFormer Fusion Model Initialized


In [78]:
for param in fusion_model.market_model.parameters():
    param.requires_grad = False

print("=" * 60)
print("Frozen TFT Parameters")
print("=" * 60)

Frozen TFT Parameters


In [79]:
# ============================================================
# Trainable Parameter Statistics
# ============================================================

total_params = sum(
    p.numel() for p in fusion_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in fusion_model.parameters()
    if p.requires_grad
)

print("=" * 60)
print("Fusion Model Statistics")
print("=" * 60)

print(f"Total Parameters      : {total_params:,}")
print(f"Trainable Parameters  : {trainable_params:,}")
print(f"Frozen Parameters     : {total_params-trainable_params:,}")

Fusion Model Statistics
Total Parameters      : 654,723
Trainable Parameters  : 0
Frozen Parameters     : 654,723


In [80]:
# ============================================================
# Fusion Forward Pass Verification
# ============================================================

dummy_market = torch.randn(
    2,
    config.SEQUENCE_LENGTH,
    config.NUM_FEATURES
).to(config.DEVICE)

dummy_news = torch.randn(
    2,
    768
).to(config.DEVICE)

with torch.no_grad():

    outputs = fusion_model(
        dummy_market,
        dummy_news
    )

print("=" * 60)
print("Fusion Forward Pass Verification")
print("=" * 60)

print("Number of Outputs :", len(outputs))

for i, output in enumerate(outputs):

    if isinstance(output, torch.Tensor):

        print(
            f"Output {i+1} : {output.shape}"
        )

Fusion Forward Pass Verification
Number of Outputs : 6
Output 1 : torch.Size([2, 3])
Output 2 : torch.Size([2])
Output 3 : torch.Size([2])
Output 4 : torch.Size([2, 4, 100, 100])
Output 5 : torch.Size([2, 128])
Output 6 : torch.Size([2, 128])


# Section 6 — Prepare Fusion Training

## Objective

This section prepares everything required for training the QuantFormer Fusion Model.

We will:

- Load cached FinBERT embeddings
- Load market labels
- Create a custom Fusion Dataset
- Build PyTorch DataLoaders
- Initialize the loss function
- Configure the optimizer

Only the Fusion layers are optimized during this stage, while the pretrained TFT encoder remains frozen.

In [81]:
import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from src.fusion.multimodal_dataset import MultiModalDataset

print("=" * 60)
print("Training Modules Imported")
print("=" * 60)

Training Modules Imported


In [83]:
# ============================================================
# Load Processed FI-2010 Dataset
# ============================================================

import numpy as np
import torch

print("=" * 60)
print("Loading Processed FI-2010 Dataset")
print("=" * 60)

X_train = np.load(config.X_TRAIN_PATH)
y_train = np.load(config.Y_TRAIN_PATH)

X_val = np.load(config.X_VAL_PATH)
y_val = np.load(config.Y_VAL_PATH)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val = torch.tensor(y_val, dtype=torch.long)

print("X_train :", X_train.shape)
print("y_train :", y_train.shape)
print("X_val   :", X_val.shape)
print("y_val   :", y_val.shape)

Loading Processed FI-2010 Dataset
X_train : torch.Size([25450, 100, 143])
y_train : torch.Size([25450])
X_val   : torch.Size([6289, 100, 143])
y_val   : torch.Size([6289])


In [84]:

train_dataset = MultiModalDataset(
    lob_features=X_train,
    news_embeddings=train_finbert_embeddings,
    labels=y_train
)

val_dataset = MultiModalDataset(
    lob_features=X_val,
    news_embeddings=val_finbert_embeddings,
    labels=y_val
)

print("=" * 60)
print("Fusion Dataset Created")
print("=" * 60)

print("Train Samples      :", len(train_dataset))
print("Validation Samples :", len(val_dataset))

Fusion Dataset Created
Train Samples      : 25450
Validation Samples : 6289
